# Unidad 4 - Clase 2
## Enmascaramiento de datos

In [1]:
-- creamos schema para los datos
CREATE SCHEMA Data;
GO
-- creamos tabla con campos con DDM
CREATE TABLE Data.Membership(
MemberID        int IDENTITY(1,1) NOT NULL PRIMARY KEY CLUSTERED,
FirstName       varchar(100) MASKED WITH (FUNCTION = 'partial(1, "xxxxx", 1)') NULL,
LastName        varchar(100) NOT NULL,
Phone           varchar(12) MASKED WITH (FUNCTION = 'default()') NULL,
Email           varchar(100) MASKED WITH (FUNCTION = 'email()') NOT NULL,
DiscountCode    smallint MASKED WITH (FUNCTION = 'random(1, 100)') NULL
);


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.016

In [2]:
-- insertamos datos de ejemplo
INSERT INTO Data.Membership (FirstName, LastName, Phone, Email, DiscountCode)
VALUES
('Roberto', 'Tamburello', '555.123.4567', 'RTamburello@contoso.com', 10),
('Janice', 'Galvin', '555.123.4568', 'JGalvin@contoso.com.co', 5),
('Shakti', 'Menon', '555.123.4570', 'SMenon@contoso.net', 50),
('Zheng', 'Mu', '555.123.4569', 'ZMu@contoso.net', 40);

(4 rows affected)

Total execution time: 00:00:00.023

In [3]:
CREATE USER MaskingTestUser WITHOUT LOGIN;
GRANT SELECT ON SCHEMA::Data TO MaskingTestUser; 

-- ejecutamos la consulta como el usuario para probar:
EXECUTE AS USER = 'MaskingTestUser';
SELECT * FROM Data.Membership;
REVERT;

(4 rows affected)

MemberID | FirstName | LastName   | Phone | Email         | DiscountCode
---------+-----------+------------+-------+---------------+-------------
1        | Rxxxxxo   | Tamburello | xxxx  | RXXX@XXXX.com | 24          
2        | Jxxxxxe   | Galvin     | xxxx  | JXXX@XXXX.com | 32          
3        | Sxxxxxi   | Menon      | xxxx  | SXXX@XXXX.com | 73          
4        | Zxxxxxg   | Mu         | xxxx  | ZXXX@XXXX.com | 16          
(4 rows)

Total execution time: 00:00:00.107

In [4]:
GRANT UNMASK TO MaskingTestUser;
EXECUTE AS USER = 'MaskingTestUser';
SELECT * FROM Data.Membership;
REVERT; 

-- Quitar permiso de UNMASK
REVOKE UNMASK TO MaskingTestUser;

(4 rows affected)

MemberID | FirstName | LastName   | Phone        | Email                   | DiscountCode
---------+-----------+------------+--------------+-------------------------+-------------
1        | Roberto   | Tamburello | 555.123.4567 | RTamburello@contoso.com | 10          
2        | Janice    | Galvin     | 555.123.4568 | JGalvin@contoso.com.co  | 5           
3        | Shakti    | Menon      | 555.123.4570 | SMenon@contoso.net      | 50          
4        | Zheng     | Mu         | 555.123.4569 | ZMu@contoso.net         | 40          
(4 rows)

Total execution time: 00:00:00.049

## Verificar usuarios

In [5]:
SELECT name,
       is_policy_checked,
       is_expiration_checked,
       LOGINPROPERTY(name, 'IsMustChange') AS IsMustChange,
       LOGINPROPERTY(name, 'IsLocked') AS IsLocked,
       LOGINPROPERTY(name, 'LockoutTime') AS LockoutTime,
       LOGINPROPERTY(name, 'PasswordLastSetTime') AS PasswordLastSetTime,
       LOGINPROPERTY(name, 'IsExpired') AS IsExpired,
       LOGINPROPERTY(name, 'BadPasswordCount') AS BadPasswordCount,
       LOGINPROPERTY(name, 'BadPasswordTime') AS BadPasswordTime,
       LOGINPROPERTY(name, 'HistoryLength') AS HistoryLength,
       modify_date
FROM sys.sql_logins;

(4 rows affected)

name                              | is_policy_checked | is_expiration_checked | IsMustChange | IsLocked | LockoutTime         | PasswordLastSetTime | IsExpired | BadPasswordCount | BadPasswordTime     | HistoryLength | modify_date            
----------------------------------+-------------------+-----------------------+--------------+----------+---------------------+---------------------+-----------+------------------+---------------------+---------------+------------------------
sa                                | 1                 | 0                     | 0            | 0        | 1900-01-01 00:00:00 | 2025-12-13 22:28:03 | 0         | 0                | 1900-01-01 00:00:00 | 1             | 2025-12-13 22:28:03.860
##MS_PolicyTsqlExecutionLogin##   | 1                 | 0                     | 0            | 0        | 1900-01-01 00:00:00 | 2026-05-04 20:28:20 | 0         | 0                | 1900-01-01 00:00:00 | 1             | 2026-05-04 20:28:20.787
##MS_Poli